In [ ]:
#cell1
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import (
    resnet50, ResNet50_Weights,
    convnext_tiny, ConvNeXt_Tiny_Weights,
    efficientnet_b2, EfficientNet_B2_Weights
)

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    roc_auc_score, confusion_matrix, classification_report
)

In [2]:
#cell 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMAGE_SIZE = 260
BATCH_SIZE = 32
NUM_WORKERS = 0
NUM_CLASSES = 3   # native output of all trained models

PROJECT_ROOT = Path("/mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI")
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

EXTERNAL_DATA_DIR = PROJECT_ROOT / "data set" / "processed"  # NORMAL / PNEUMONIA

# native class order used during training: index 0=BACTERIA, 1=NORMAL, 2=VIRUS
NATIVE_CLASS_NAMES = ["BACTERIA", "NORMAL", "VIRUS"]
# map native prediction -> external binary label (0=NORMAL, 1=PNEUMONIA)
NATIVE_TO_BINARY = {0: 1, 1: 0, 2: 1}

CHECKPOINTS = {
    "PneumoXNet":      MODELS_DIR / "pneumoxnet_seed42_best_acc.pth",
    "EfficientNet-B2":  MODELS_DIR / "efficientnetb2_seed42_best.pth",
    "ResNet-50":        MODELS_DIR / "resnet50_seed42_best.pth",
    "ConvNeXt-Tiny":    MODELS_DIR / "convnexttiny_seed42_best.pth",
}

for name, path in CHECKPOINTS.items():
    print(f"{name:<16}: {path} | exists: {path.exists()}")

PneumoXNet      : /mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI/models/pneumoxnet_seed42_best_acc.pth | exists: True
EfficientNet-B2 : /mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI/models/efficientnetb2_seed42_best.pth | exists: True
ResNet-50       : /mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI/models/resnet50_seed42_best.pth | exists: True
ConvNeXt-Tiny   : /mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI/models/convnexttiny_seed42_best.pth | exists: True
